# TTZ v3 推筒子 AI 訓練

基於 YOLOv8n，訓練推筒子牌面辨識模型

**類別順序**: `0=bai, 1=t1, 2=t2, 3=t3, 4=t4, 5=t5, 6=t6, 7=t7, 8=t8, 9=t9`

In [ ]:
# 1. 安裝依賴
!pip install ultralytics -q

In [ ]:
# 2. 下載並準備資料集
import os

# 從 GitHub repo 下載整個專案
!wget -q https://github.com/Powerck7788/ttz_v3_dataset/archive/refs/heads/main.zip
!unzip -q -o main.zip
!mv ttz_v3_dataset-main mahjong_v3_dataset

# 解壓資料集 zip
!unzip -q -o mahjong_v3_dataset/ttz_v3_dataset.zip -d mahjong_v3_dataset

DST = '/content/mahjong_v3_dataset'
print(f"資料集已準備好：{DST}")
!ls -la $DST
!ls -la $DST/yolo_dataset/images/

In [ ]:
# 3. 使用內建的 data.yaml
!cat /content/mahjong_v3_dataset/data.yaml

In [ ]:
# 4. 開始訓練
from ultralytics import YOLO

# 載入預訓練模型
model = YOLO('yolov8n.pt')

# 訓練參數
results = model.train(
    data='/content/mahjong_v3_dataset/data.yaml',
    epochs=200,
    patience=30,
    batch=32,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.005,
    device=0,
    project='/content/ttz_v3_training',
    name='v3_run1',
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=10,
)

print("訓練完成！")

In [ ]:
# 5. 導出為 ONNX (部署到 Pi)
from ultralytics import YOLO
import os

# 載入最佳模型
best_model_path = '/content/ttz_v3_training/v3_run1/weights/best.pt'
model = YOLO(best_model_path)

# 導出為 ONNX (imgsz=160, FP16)
model.export(
    format='onnx',
    imgsz=160,
    half=True,
    simplify=True,
    opset=12,
    optimize=True
)

print(f"\n模型已導出：{best_model_path.replace('.pt', '.onnx')}")

# 顯示檔案大小
onnx_path = best_model_path.replace('.pt', '.onnx')
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f"ONNX 模型大小：{size_mb:.2f} MB")

In [ ]:
# 6. 下載訓練結果
from google.colab import files
import zipfile
import os

# 壓縮最佳模型
best_onnx = '/content/ttz_v3_training/v3_run1/weights/best.onnx'
best_pt = '/content/ttz_v3_training/v3_run1/weights/best.pt'

with zipfile.ZipFile('ttz_v3_model.zip', 'w') as zf:
    if os.path.exists(best_onnx):
        zf.write(best_onnx, 'best.onnx')
    if os.path.exists(best_pt):
        zf.write(best_pt, 'best.pt')
    results_path = '/content/ttz_v3_training/v3_run1/results.csv'
    if os.path.exists(results_path):
        zf.write(results_path, 'results.csv')

files.download('ttz_v3_model.zip')
print("訓練結果已打包下載！")

In [ ]:
# 7. 驗證模型（可選）
from ultralytics import YOLO

model_onnx = YOLO(best_onnx)
metrics = model_onnx.val(data='/content/mahjong_v3_dataset/data.yaml')

print(f"\n驗證結果:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")